# GNN Node-Targeting — Attention GNN Training

This notebook trains the `AttentionSceneGNN` directly using the newly integrated `svgpatchlab.gnn` package.

In [ ]:
import subprocess, sys, os
from pathlib import Path

GITHUB_TOKEN = "PASTE_YOUR_NEW_TOKEN_HERE"
GITHUB_USER = "smerarawal"
REPO_NAME = "EditSVG-patch-lab"

os.chdir("/kaggle/working")
if not Path(REPO_NAME).exists():
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--recurse-submodules", clone_url], check=True)
os.chdir(REPO_NAME)
REPO_ROOT = Path.cwd()

# Install the new GNN dependencies!
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[vision,gnn]", "--quiet"], check=True)

if not Path("SVGEditBench").exists() or not list(Path("SVGEditBench").glob("*")):
    subprocess.run(["git", "clone", "https://github.com/mti-lab/SVGEditBench.git"], check=True)

print("\u2713 Setup complete. Repo root:", REPO_ROOT)

In [ ]:
import sys
sys.path.insert(0, ".")
from svgpatchlab.core import build_scene
from svgpatchlab.core.patch import derive_patch, PatchError
from svgpatchlab.data import SVGEditBench

TARGETING_TASKS = ["change_color", "set_contour"]

bench = SVGEditBench("SVGEditBench")
triples = []
multi_target_count = 0
derive_failed_count = 0
total = 0

for case in bench.iter_cases(tasks=TARGETING_TASKS, limit_per_task=100):
    total += 1
    try:
        gold_patch = derive_patch(case.source_svg, case.answer_svg)
    except PatchError as e:
        derive_failed_count += 1
        continue

    targets = [t for op in gold_patch.operations for t in op.targets]
    if len(targets) != 1:
        multi_target_count += 1
        continue

    scene = build_scene(case.source_svg)
    triples.append((case.instruction, scene, targets[0], case.source_svg))

print(f"Total cases scanned: {total}")
print(f"Usable single-target triples: {len(triples)}")

In [ ]:
# Import directly from the new gnn package!
from svgpatchlab.gnn import scene_to_graph_inputs, get_text_encoder
import torch
from torch_geometric.data import Data

text_encoder = get_text_encoder()
INSTRUCTION_EMBED_DIM = text_encoder.get_sentence_embedding_dimension()

raw_graphs = []
for instruction, scene, target_id, source_svg in triples:
    # Now scene_to_graph_inputs does all the heavy lifting
    x, edge_index, edge_type, id_to_idx, vision_scalar_x, vision_summaries = scene_to_graph_inputs(scene)
    if target_id not in id_to_idx:
        continue
    raw_graphs.append((x, edge_index, edge_type, id_to_idx, vision_scalar_x, vision_summaries, target_id, instruction))

instruction_texts = [g[7] for g in raw_graphs]
instruction_embeds = text_encoder.encode(instruction_texts, convert_to_tensor=True, show_progress_bar=True)

all_vision_summaries = []
for g in raw_graphs:
    all_vision_summaries.extend(g[5])

vision_summary_embeds = text_encoder.encode(all_vision_summaries, convert_to_tensor=True) if all_vision_summaries else torch.zeros((0, INSTRUCTION_EMBED_DIM))

pyg_dataset = []
offset = 0
for (x, edge_index, edge_type, id_to_idx, vision_scalar_x, vision_summaries, target_id, instruction), instr_embed in zip(raw_graphs, instruction_embeds):
    n = x.size(0)
    node_vision_embeds = vision_summary_embeds[offset:offset + n].to(x.device)
    offset += n
    has_vision = vision_scalar_x[:, 0]
    vision_x = torch.cat([vision_scalar_x, node_vision_embeds], dim=1)
    instr_graph_level = instr_embed.to(x.device).unsqueeze(0)
    y = torch.tensor([id_to_idx[target_id]], dtype=torch.long)
    pyg_dataset.append(Data(
        x=x, edge_index=edge_index, edge_type=edge_type,
        instr=instr_graph_level, vision_x=vision_x, has_vision=has_vision,
        y=y, num_nodes=n,
    ))

print(f"Built {len(pyg_dataset)} graph examples")

In [ ]:
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
import random

# Import the new Attention GNN
from svgpatchlab.gnn import AttentionSceneGNN, VISION_SCALAR_DIM

random.seed(0)
random.shuffle(pyg_dataset)
split = int(len(pyg_dataset) * 0.8)
train_data, val_data = pyg_dataset[:split], pyg_dataset[split:]

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

in_dim = pyg_dataset[0].x.size(1)
instr_dim = pyg_dataset[0].instr.size(1)
vision_dim = pyg_dataset[0].vision_x.size(1)

# Use the upgraded Attention Model
model = AttentionSceneGNN(in_dim, instr_dim, vision_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            scores = model(batch.x, batch.edge_index, batch.edge_type, batch.instr, batch.vision_x, batch.has_vision, batch.batch)
            for i in range(batch.num_graphs):
                mask = batch.batch == i
                pred = scores[mask].argmax().item()
                gold = batch.y[i].item()
                correct += int(pred == gold)
                total += 1
    return correct / total if total else 0.0

for epoch in range(30):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        scores = model(batch.x, batch.edge_index, batch.edge_type, batch.instr, batch.vision_x, batch.has_vision, batch.batch)
        loss = 0.0
        for i in range(batch.num_graphs):
            mask = batch.batch == i
            node_scores = scores[mask].unsqueeze(0)
            gold = batch.y[i].unsqueeze(0)
            loss = loss + F.cross_entropy(node_scores, gold)
        loss = loss / batch.num_graphs
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0 or epoch == 29:
        val_acc = evaluate(val_loader)
        print(f"Epoch {epoch:2d}  train_loss={total_loss/len(train_loader):.4f}  val_top1_acc={val_acc*100:.1f}%")

torch.save(model.state_dict(), "gnn_node_targeting.pt")
print("\u2713 Saved gnn_node_targeting.pt")